In [ ]:
# ---------------
# Dependencies
# ---------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score

In [ ]:
# ------------------------------------
# Device Setup (Is CUDA available?)
# ------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", DEVICE)

Using device:  cuda


In [ ]:
# -----------------------------------
# Load Dataset from GDrive (Colab)
# -----------------------------------

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------------
# Load dataset
# ---------------

df = pandas.read_csv("/content/drive/MyDrive/data/dataset_original.csv")

if "hash" in df.columns:
  df = df.drop(columns=["hash"])

# Balanced Dataset:
# df_majority = df[df["malware"] == 1]
# df_minority = df[df["malware"] == 0]

# df_majority_down = df_majority.sample(n=len(df_minority), random_state=42)
# df_balanced = pandas.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

# df = df_balanced

In [ ]:
# -------------
# Data Setup
# -------------

X = df.drop(columns=['malware']).values.astype(numpy.float32)
y = df['malware'].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df['malware'], random_state=42
)

X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)

y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

In [ ]:
# ----------------------------------------------------------
# Create training data loader with weighted class samples
# ----------------------------------------------------------

# class_sample_counts = numpy.bincount(y_train.cpu().numpy())
# weights = 1. / class_sample_counts
# sample_weights = weights[y_train.cpu().numpy()]
# sample_weights = torch.from_numpy(sample_weights).float()

# sampler = WeightedRandomSampler(
#     sample_weights,
#     num_samples=len(sample_weights),
#     replacement=True
# )

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
    # sampler=sampler
)

In [ ]:
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
LATENT_DIM = 128
EMB_DIM = 128
BATCH_SIZE = 128
EPOCHS = 100

In [ ]:
class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()

    self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)

    self.conv = nn.Sequential(
        nn.Conv1d(EMB_DIM, 128, 5, padding=2),
        nn.LeakyReLU(0.2),
        nn.Conv1d(128, 256, 5, padding=2),
        nn.LeakyReLU(0.2),
        nn.AdaptiveMaxPool1d(1)
    )

    self.fc = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256, 128),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.4),
        nn.Linear(128, NUM_CLASSES + 1)
    )

  def forward(self, x, embedded=False):
    if not embedded:
        x = x.long()
        x = self.embedding(x)

    x = x.permute(0, 2, 1) # (BatchSize, EmbeddingSpace, SequenceLength)
    x = self.conv(x)
    x = self.fc(x)

    return x

In [ ]:
class Generator(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(LATENT_DIM, 256),
        nn.ReLU(),
        nn.Linear(256, SEQ_LEN * VOCAB_SIZE)
    )

  def forward(self, z, temperature=0.5):
    logits = self.net(z)
    logits = logits.view(-1, SEQ_LEN, VOCAB_SIZE)
    gumbel = F.gumbel_softmax(logits, tau=temperature, hard=False)
    return gumbel

In [ ]:
D = Discriminator().to(DEVICE)
G = Generator().to(DEVICE)

optimizer_D = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_G = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

In [ ]:
# best_f1 = 0
# patience = 30
# patience_counter = 0

for epoch in range(EPOCHS):

  D.train()
  G.train()

  for real_x, real_y in train_loader:

      real_x = real_x.to(DEVICE)
      real_y = real_y.to(DEVICE)
      batch_size = real_x.size(0)

      """Discriminator Training"""
      optimizer_D.zero_grad()

      # Real
      logits_real = D(real_x)
      loss_real = F.cross_entropy(
          logits_real,
          real_y
      )

      # Fake
      z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
      fake_probs = G(z)

      # Soft tokens to embedding space
      fake_emb = torch.matmul(
          fake_probs,
          D.embedding.weight
      )

      logits_fake = D(fake_emb, embedded=True)

      fake_labels = torch.full(
          (batch_size,),
          NUM_CLASSES,
          device=DEVICE
      )

      loss_fake = F.cross_entropy(
          logits_fake,
          fake_labels
      )

      loss_D = loss_real + loss_fake
      loss_D.backward()
      optimizer_D.step()

      """Generator Training"""
      optimizer_G.zero_grad()

      z = torch.randn(batch_size, LATENT_DIM).to(DEVICE)
      fake_probs = G(z)

      fake_emb = torch.matmul(
          fake_probs,
          D.embedding.weight
      )

      logits_fake = D(fake_emb, embedded=True)

      probs = F.softmax(logits_fake, dim=1)
      loss_G = -torch.mean(torch.log(1 - probs[:, NUM_CLASSES] + 1e-8))

      loss_G.backward()
      optimizer_G.step()

  # """Validation"""
  # D.eval()
  # with torch.no_grad():

  #     logits = D(X_val)
  #     probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
  #     preds = torch.argmax(probs, dim=1)

  #     val_f1 = f1_score(
  #         y_val.cpu().numpy(),
  #         preds.cpu().numpy(),
  #         average="macro"
  #     )

  #     val_auc = roc_auc_score(
  #         y_val.cpu().numpy(),
  #         probs[:,1].cpu().numpy()
  #     )

  print(f"Epoch {epoch+1} | D {loss_D:.4f} | G {loss_G:.4f}") #| Macro-F1 {val_f1:.4f} | AUC {val_auc:.4f})

  # if val_f1 > best_f1:
  #     best_f1 = val_f1
  #     patience_counter = 0
  #     torch.save(D.state_dict(), "best_sgan.pt")
  # else:
  #     patience_counter += 1

  # if patience_counter > patience:
  #     print("Early stopping.")
  #     break

Epoch 1 | D 0.0702 | G 4.8162
Epoch 2 | D 0.0161 | G 6.2809
Epoch 3 | D 0.0136 | G 6.9068
Epoch 4 | D 0.0074 | G 7.9826
Epoch 5 | D 0.0102 | G 8.3185
Epoch 6 | D 0.0162 | G 9.0398
Epoch 7 | D 0.0099 | G 8.6517
Epoch 8 | D 0.0024 | G 9.0412
Epoch 9 | D 0.0014 | G 9.1979
Epoch 10 | D 0.0051 | G 9.2462
Epoch 11 | D 0.0008 | G 9.3325
Epoch 12 | D 0.0026 | G 9.0597
Epoch 13 | D 0.0018 | G 9.8174
Epoch 14 | D 0.0026 | G 10.1011
Epoch 15 | D 0.0006 | G 10.3792
Epoch 16 | D 0.0016 | G 10.8814
Epoch 17 | D 0.0004 | G 10.3343
Epoch 18 | D 0.0004 | G 10.2484
Epoch 19 | D 0.0003 | G 10.9598
Epoch 20 | D 0.0009 | G 11.0881
Epoch 21 | D 0.0014 | G 10.8447
Epoch 22 | D 0.0005 | G 12.3564
Epoch 23 | D 0.0008 | G 12.5401
Epoch 24 | D 0.0008 | G 11.7568
Epoch 25 | D 0.0009 | G 10.9136
Epoch 26 | D 0.0006 | G 12.6605
Epoch 27 | D 0.0019 | G 11.2562
Epoch 28 | D 0.0010 | G 11.5965
Epoch 29 | D 0.0014 | G 14.1125
Epoch 30 | D 0.0012 | G 12.8563
Epoch 31 | D 0.0013 | G 13.8952
Epoch 32 | D 0.0003 | G 14.857

In [ ]:
torch.save(D.state_dict(), "sgan_imb.pt")

In [ ]:
D.load_state_dict(torch.load("sgan_imb.pt"))
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits[:, :NUM_CLASSES], dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(
    y_val.cpu().numpy(),
    preds.cpu().numpy(),
    digits=4
))

print("PR-AUC:",
      average_precision_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

print("ROC-AUC:",
      roc_auc_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

              precision    recall  f1-score   support

           0     0.9749    0.8395    0.9022       324
           1     0.9960    0.9995    0.9977     12839

    accuracy                         0.9955     13163
   macro avg     0.9854    0.9195    0.9499     13163
weighted avg     0.9954    0.9955    0.9954     13163

PR-AUC: 0.9998288936944241
ROC-AUC: 0.9942187624704435


# Ablation

In [1]:
# ---------------
# Dependencies
# ---------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import pandas
import numpy
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

from google.colab import drive
drive.mount('/content/drive')

df = pandas.read_csv("/content/drive/MyDrive/data/dataset_original.csv")

if "hash" in df.columns:
    df = df.drop(columns=["hash"])

X = df.drop(columns=['malware']).values.astype(numpy.float32)
y = df['malware'].values.astype(numpy.int64)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.30, stratify=df['malware'], random_state=42
)

X_train = torch.tensor(X_train).to(DEVICE)
X_val = torch.tensor(X_val).to(DEVICE)
y_train = torch.tensor(y_train).to(DEVICE)
y_val = torch.tensor(y_val).to(DEVICE)

# class_sample_counts = numpy.bincount(y_train.cpu().numpy())
# class_weights = 1. / class_sample_counts
# class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
    # shuffle=True
)

# Hyperparameters
VOCAB_SIZE = 307
SEQ_LEN = 100
NUM_CLASSES = 2
EMB_DIM = 128
EPOCHS = 100

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMB_DIM)

        self.conv = nn.Sequential(
            nn.Conv1d(EMB_DIM, 128, 5, padding=2),
            nn.LeakyReLU(0.2),
            nn.Conv1d(128, 256, 5, padding=2),
            nn.LeakyReLU(0.2),
            nn.AdaptiveMaxPool1d(1)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        x = x.long()
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        x = self.fc(x)
        return x


D = Discriminator().to(DEVICE)

optimizer = optim.Adam(D.parameters(), lr=2e-4)

# Training
for epoch in range(EPOCHS):

    D.train()
    total_loss = 0

    for real_x, real_y in train_loader:

        optimizer.zero_grad()

        logits = D(real_x)

        loss = F.cross_entropy(
            logits,
            real_y,
            # weight=class_weights
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


torch.save(D.state_dict(), "d_only.pt")

# Evaluation
D.eval()

with torch.no_grad():
    logits = D(X_val)
    probs = F.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)

print(classification_report(
    y_val.cpu().numpy(),
    preds.cpu().numpy(),
    digits=4
))

print("PR-AUC:",
      average_precision_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

print("ROC-AUC:",
      roc_auc_score(
          y_val.cpu().numpy(),
          probs[:,1].cpu().numpy()
      ))

Using device: cuda
Mounted at /content/drive
Epoch 1 | Loss: 0.1029
Epoch 2 | Loss: 0.0380
Epoch 3 | Loss: 0.0197
Epoch 4 | Loss: 0.0138
Epoch 5 | Loss: 0.0107
Epoch 6 | Loss: 0.0092
Epoch 7 | Loss: 0.0088
Epoch 8 | Loss: 0.0082
Epoch 9 | Loss: 0.0075
Epoch 10 | Loss: 0.0071
Epoch 11 | Loss: 0.0059
Epoch 12 | Loss: 0.0059
Epoch 13 | Loss: 0.0053
Epoch 14 | Loss: 0.0055
Epoch 15 | Loss: 0.0048
Epoch 16 | Loss: 0.0048
Epoch 17 | Loss: 0.0049
Epoch 18 | Loss: 0.0045
Epoch 19 | Loss: 0.0045
Epoch 20 | Loss: 0.0049
Epoch 21 | Loss: 0.0045
Epoch 22 | Loss: 0.0040
Epoch 23 | Loss: 0.0035
Epoch 24 | Loss: 0.0038
Epoch 25 | Loss: 0.0040
Epoch 26 | Loss: 0.0041
Epoch 27 | Loss: 0.0038
Epoch 28 | Loss: 0.0035
Epoch 29 | Loss: 0.0034
Epoch 30 | Loss: 0.0035
Epoch 31 | Loss: 0.0036
Epoch 32 | Loss: 0.0034
Epoch 33 | Loss: 0.0034
Epoch 34 | Loss: 0.0033
Epoch 35 | Loss: 0.0033
Epoch 36 | Loss: 0.0033
Epoch 37 | Loss: 0.0036
Epoch 38 | Loss: 0.0032
Epoch 39 | Loss: 0.0062
Epoch 40 | Loss: 0.0051
Epoc